# Tratamento de Dados com PySpark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, sum
from pyspark.sql.types import IntegerType, DateType

spark = SparkSession.builder.appName("Tratamento").getOrCreate()

## Leitura dos arquivos

In [ ]:
df_video = spark.read.csv("videos-stats.csv", header=True, inferSchema=True)
df_comentario = spark.read.csv("comments.csv", header=True, inferSchema=True)

## Tratamento de nulos

In [ ]:
df_video = df_video.fillna({
    "Likes": 0,
    "Comments": 0,
    "Views": 0
})

## Contagem inicial

In [ ]:
print("Videos:", df_video.count())
print("Comentarios:", df_comentario.count())

## Remover nulos em Video ID

In [ ]:
df_video = df_video.dropna(subset=["Video ID"])
df_comentario = df_comentario.dropna(subset=["Video ID"])

## Contagem após limpeza

In [ ]:
print("Videos:", df_video.count())
print("Comentarios:", df_comentario.count())

## Remover duplicados

In [ ]:
df_video = df_video.dropDuplicates(["Video ID"])

## Conversão de tipos

In [ ]:
df_video = df_video.withColumn("Likes", col("Likes").cast(IntegerType()))                    .withColumn("Comments", col("Comments").cast(IntegerType()))                    .withColumn("Views", col("Views").cast(IntegerType()))

In [ ]:
df_comentario = df_comentario.withColumn("Likes Comment", col("Likes").cast(IntegerType()))                              .withColumn("Sentiment", col("Sentiment").cast(IntegerType()))                              .drop("Likes")

## Criar coluna Interaction

In [ ]:
df_video = df_video.withColumn(
    "Interaction",
    col("Likes") + col("Comments") + col("Views")
)

## Data e Ano

In [ ]:
df_video = df_video.withColumn("Published At", col("Published At").cast(DateType()))
df_video = df_video.withColumn("Year", year(col("Published At")))

## Join com comentários

In [ ]:
df_join_video_comments = df_video.join(df_comentario, on="Video ID", how="inner")
df_join_video_comments.show(5)

## Ler USvideos

In [ ]:
df_us_videos = spark.read.csv("USvideos.csv", header=True, inferSchema=True)

## Join com USvideos

In [ ]:
df_join_video_usvideos = df_video.join(df_us_videos, on="Title", how="inner")
df_join_video_usvideos.show(5)

## Contar nulos

In [ ]:
df_video.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_video.columns
]).show()

## Salvar parquet

In [ ]:
df_video = df_video.drop("_c0")
df_video.write.mode("overwrite").parquet("videos-tratados-parquet")

In [ ]:
df_join_video_comments = df_join_video_comments.drop("_c0")
df_join_video_comments.write.mode("overwrite").parquet("videos-comments-tratados-parquet")